# Phần 1. NumPy trong workflow ML/DL

Các bài dưới đây dùng dữ liệu nhỏ để mô phỏng preprocessing, inference và xử lý
tensor trong một pipeline thực tế.

In [ ]:
STUDENT_NAME = "Lê Nguyễn Loan Thảo"  # TODO: Họ và tên
STUDENT_ID = "2413194"    # TODO: MSSV

print(f"Student: {STUDENT_NAME} ({STUDENT_ID})")

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams["figure.figsize"] = (8, 4.8)

DATA_CANDIDATES = [
    Path("week02/numpy-pandas-eda-hw/data/automobile_raw.csv"),
    Path("data/automobile_raw.csv"),
    Path("../data/automobile_raw.csv"),
]
DATA_PATH = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Không tìm thấy data/automobile_raw.csv")

print("Data path:", DATA_PATH.resolve())

## N1. Stable softmax cho batch logits

Một classifier trả về `logits` có shape `(batch_size, num_classes)`. Tính softmax
theo từng mẫu bằng cách trừ giá trị lớn nhất trên mỗi hàng trước khi gọi `np.exp`.
Cách viết này tránh overflow khi logits có giá trị lớn.

**Biến đầu ra bắt buộc**

- `shifted_logits`: logits sau khi trừ row-wise maximum.
- `class_probabilities`: xác suất mỗi class, mỗi hàng có tổng bằng 1.
- `predicted_classes`: class có xác suất lớn nhất của từng mẫu.
- `confidence_scores`: xác suất lớn nhất của từng mẫu.

In [ ]:
logits = np.array([
    [2.0, 1.0, 0.1],
    [1000.0, 1001.0, 999.0],
    [-2.0, -1.0, 3.0],
    [0.5, 0.5, 0.5],
], dtype=np.float64)

In [ ]:
# TODO N1
shifted_logits = logits - np.max(logits, axis=1, keepdims=True)
exp_logits = np.exp(shifted_logits)
class_probabilities = exp_logits / np.sum(exp_logits, axis=1, keepdims=True)
predicted_classes = np.argmax(class_probabilities, axis=1)
confidence_scores = np.max(class_probabilities, axis=1)

In [ ]:
required = [
    "shifted_logits",
    "class_probabilities",
    "predicted_classes",
    "confidence_scores",
]
if not all(name in globals() for name in required):
    print("Complete N1 to run this self-check.")
else:
    assert class_probabilities.shape == logits.shape
    assert np.all(np.isfinite(class_probabilities))
    assert np.allclose(class_probabilities.sum(axis=1), 1.0)
    assert predicted_classes.shape == (logits.shape[0],)
    assert confidence_scores.shape == (logits.shape[0],)
    print("N1 self-check passed")

## N2. Chuẩn hóa train và validation không gây leakage

Mỗi hàng là một mẫu, mỗi cột là một feature. Tính mean/std **chỉ từ `X_train`**,
sau đó dùng cùng thống kê để transform cả train và validation.

**Biến đầu ra bắt buộc**

- `train_feature_mean`, `train_feature_std`: shape `(4,)`.
- `X_train_scaled`: train set đã chuẩn hóa.
- `X_val_scaled`: validation set dùng thống kê từ train.

In [ ]:
# Features: height_cm, weight_kg, activity_hours, age
X_train = np.array([
    [170.0, 65.0, 1.2, 22.0],
    [180.0, 80.0, 2.4, 35.0],
    [160.0, 50.0, 0.8, 19.0],
    [175.0, 70.0, 1.5, 28.0],
    [168.0, 60.0, 1.0, 24.0],
    [182.0, 90.0, 3.0, 41.0],
])

X_val = np.array([
    [172.0, 68.0, 1.4, 26.0],
    [190.0, 95.0, 3.4, 45.0],
])

In [ ]:
# TODO N2
train_feature_mean = np.mean(X_train, axis=0)
train_feature_std = np.std(X_train, axis=0)

X_train_scaled = (X_train - train_feature_mean) / train_feature_std
X_val_scaled = (X_val - train_feature_mean) / train_feature_std

In [ ]:
required = [
    "train_feature_mean",
    "train_feature_std",
    "X_train_scaled",
    "X_val_scaled",
]
if not all(name in globals() for name in required):
    print("Complete N2 to run this self-check.")
else:
    assert X_train_scaled.shape == X_train.shape
    assert X_val_scaled.shape == X_val.shape
    assert np.allclose(X_train_scaled.mean(axis=0), 0.0)
    assert np.allclose(X_train_scaled.std(axis=0), 1.0)
    print("N2 self-check passed")

## N3. Tạo review queue sau inference

Giả sử `class_probabilities` đến từ N1. Một prediction cần được kiểm tra thủ công
nếu dự đoán sai **hoặc** confidence nhỏ hơn `0.70`.

**Biến đầu ra bắt buộc**

- `correct_mask`
- `high_confidence_mask`
- `review_mask`
- `review_indices`

In [ ]:
true_labels = np.array([0, 2, 2, 1])
confidence_threshold = 0.70

In [ ]:
# TODO N3
correct_mask = predicted_classes == true_labels
high_confidence_mask = confidence_scores >= confidence_threshold
review_mask = (~correct_mask) | (~high_confidence_mask)
review_indices = np.where(review_mask)[0]

## N4. Tiền xử lý và augment một batch ảnh

`image_batch_uint8` có layout `(B, H, W, C)`. Chuyển batch về `float32` trong đoạn
`[0, 1]`, sau đó tạo một batch mới được flip ngang. Batch augment phải có bộ nhớ
độc lập để việc chỉnh sửa không làm thay đổi batch đã normalize.

Sau khi tạo batch augment, đặt pixel `augmented_batch[0, 0, 0, 0] = 1.0`.

**Biến đầu ra bắt buộc:** `normalized_batch`, `augmented_batch`.

In [ ]:
image_batch_uint8 = (
    np.arange(2 * 4 * 4 * 3, dtype=np.uint8)
    .reshape(2, 4, 4, 3)
)

In [ ]:
# TODO N4
normalized_batch = image_batch_uint8.astype(np.float32) / 255.0
augmented_batch = normalized_batch[:, :, ::-1, :].copy()
augmented_batch[0, 0, 0, 0] = 1.0

# Phần 2. EDA với Automobile

Đọc `data/data_dictionary.md` trước khi xử lý.

## Câu hỏi mở đầu

1. Mỗi dòng đại diện cho đối tượng gì?
2. Ký hiệu missing value trong CSV là gì?
3. `symboling` có ý nghĩa gì?

**Trả lời**
1. Đại diện cho một mẫu xe trong tập dữ liệu.
2. Dấu ?
3. Là chỉ số đánh giá mức độ rủi ro bảo hiệm xe. Giá trị càng lớn thì mức độ rủi ro càng cao.

## D1. Load và inspect raw CSV

Load dữ liệu sao cho dấu `?` vẫn là chuỗi để quan sát ảnh hưởng tới dtype.

**Biến đầu ra bắt buộc**

- `raw_df`: DataFrame raw.
- `raw_shape`: tuple.
- `raw_missing_marker_count`: tổng số dấu `?`.

In [ ]:
# TODO D1
raw_df = pd.read_csv(DATA_PATH, keep_default_na=False)
raw_shape = raw_df.shape
raw_missing_marker_count = (raw_df == "?").sum().sum()

## D2. Missing values và dtype

1. Thay `?` bằng `np.nan`.
2. Chuyển các cột trong `NUMERIC_COLUMNS` bằng `pd.to_numeric`.
3. Tạo báo cáo missing.

**Biến đầu ra bắt buộc:** `df_clean`, `missing_by_column`.

In [ ]:
NUMERIC_COLUMNS = ['symboling', 'normalized_losses', 'wheel_base', 'length', 'width', 'height', 'curb_weight', 'engine_size', 'bore', 'stroke', 'compression_ratio', 'horsepower', 'peak_rpm', 'city_mpg', 'highway_mpg', 'price']

In [ ]:
# TODO D2
df_clean = raw_df.replace("?", np.nan)
for column in NUMERIC_COLUMNS:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )
missing_by_column = df_clean.isna().sum()

### Giải thích cách làm sạch dữ liệu

- Vì sao không nên fill tất cả numeric columns bằng cùng một giá trị?
- Với `price`, lựa chọn drop hay fill phù hợp hơn cho bài EDA này? Vì sao?
- `normalized_losses` thiếu nhiều dữ liệu hơn các cột khác. Điều này ảnh hưởng thế nào?

**Nhận xét**
Không nên điền tất cả các cột số bằng cùng một giá trị vì mỗi cột có ý nghĩa và phân phối khác nhau, việc điền cùng một loại giá trị có thể làm sai lệch dự liệu. 
Đối với cột price, việc loại bỏ các dòng bị thiếu phù hợp hơn vì đây là biến mục tiêu chính của bài. 
Cột normalized_losses có số lượng giá trị thiếu khá lớn nên các phân tích liên quan đến cột này có thể kém tin cậy hơn. Ngoài ra, việc thiếu dữ liệu nhiều cũng có thể làm giảm số lượng mẫu sử dụng được trong quá trình phân tích.

## D3. DataFrame sang NumPy

Dùng sáu cột trong `AUTO_FEATURES`. Drop các dòng thiếu ít nhất một trong
sáu cột, sau đó chuyển sang `float64` NumPy array và chuẩn hóa theo feature.

**Biến đầu ra bắt buộc**

- `analysis_df`
- `X_auto`
- `auto_feature_mean`
- `auto_feature_std`
- `X_auto_scaled`

In [ ]:
AUTO_FEATURES = ['curb_weight', 'engine_size', 'horsepower', 'city_mpg', 'highway_mpg', 'price']

In [ ]:
# TODO D3
analysis_df = df_clean[AUTO_FEATURES].dropna()
X_auto = analysis_df.to_numpy(dtype=np.float64)
auto_feature_mean = np.mean(X_auto, axis=0)
auto_feature_std = np.std(X_auto, axis=0)
X_auto_scaled = (
    X_auto - auto_feature_mean
) / auto_feature_std

## D4. Outlier theo price z-score

Tính z-score của `price` bằng NumPy. Một dòng được xem là outlier trong bài
này khi `abs(z) > 2`.

**Biến đầu ra bắt buộc:** `price_z`, `price_outlier_mask`, `price_outliers`.

In [ ]:
# TODO D4
price_index = AUTO_FEATURES.index("price")
price_z = (
    X_auto[:, price_index] - auto_feature_mean[price_index]
) / auto_feature_std[price_index]
price_outlier_mask = np.abs(price_z) > 2
price_outliers = analysis_df.loc[price_outlier_mask]

## D5. Correlation và GroupBy

**Biến đầu ra bắt buộc**

- `engine_price_corr`: Pearson correlation tính bằng NumPy.
- `price_by_body_style`: Series mean price theo `body_style`, sort index.

In [ ]:
# TODO D5
engine_price_corr = np.corrcoef(
    analysis_df["engine_size"],
    analysis_df["price"]
)[0, 1]
price_by_body_style = (
    df_clean
    .dropna(subset=["price"])
    .groupby("body_style")["price"]
    .mean()
    .sort_index()
)

# Phần 3. Visualization và insight

Mỗi biểu đồ cần:

1. một câu hỏi;
2. title, axis labels và unit;
3. lựa chọn chart phù hợp;
4. 1--2 câu nhận xét ngay dưới chart.

## M2.1 Price phân phối như thế nào?

In [ ]:
# TODO M2.1: histogram/KDE của price
plt.figure(figsize=(8,5))
sns.histplot(df_clean["price"].dropna(), bins=30, kde=True)
plt.title('Distribution of Car Price')
plt.xlabel('Price (USD)')
plt.ylabel('Frequency (cars)')
plt.show()

**Nhận xét:** 
Dataset lệch phải khi hầu hết xe nằm ở mức giá thấp và trung bình, còn lại số ít xe giá rất cao. 
Ngoài ra biểu đồ còn cho thấy 1 số giá trị ngoại lệ ở bên phải.

## M2.2 Dataset có cân bằng theo body style không?

In [ ]:
# TODO M2.2: countplot của body_style
plt.figure(figsize=(8,5))
sns.countplot(data=df_clean, x='body_style')
plt.title('Number of Cars by Body Style')
plt.xlabel('Body Style')
plt.ylabel('Count (cars)')
plt.xticks(rotation=45)
plt.show()

**Nhận xét:** 
Dataset không hoàn toàn cân bằng, một số xuất hiện rất nhiều nhưng một số nhóm lại rất ít mẫu.

## M2.3 Price khác nhau theo body style ra sao?

In [ ]:
# TODO M2.3: boxplot price theo body_style
plt.figure(figsize=(8,5))
sns.boxplot(data=df_clean, x='body_style', y='price')
plt.title('Price Distribution by Body Style')
plt.xlabel('Body Style')
plt.ylabel('Price (USD)')
plt.xticks(rotation=45)
plt.show()

**Nhận xét:** 
Giá xe có sự khác biệt rõ ràng, một số nhóm có median price cao hơn và độ biến động giá lớn hơn trong khi số còn lại thấp hơn hẳn => kiểu thân xe là một yếu tố ảnh hưởng đến giá bán.

## M2.4 Engine size liên quan thế nào tới price?

In [ ]:
# TODO M2.4: scatterplot engine_size và price, hue=fuel_type
plt.figure(figsize=(8,5))
sns.scatterplot(
    data=df_clean,
    x='engine_size',
    y='price',
    hue='fuel_type'
)
plt.title('Engine Size vs Price')
plt.xlabel('Engine Size')
plt.ylabel('Price (USD)')
plt.show()

**Nhận xét:** 
Khi engine size tăng, giá xe cũng có xu hướng tăng, cho thấy mối quan hệ tương quan dương giữa 2 biến. 
Ngoài ra vẫn còn 1 số điểm phân tán khác, chứng tỏ giá xe còn phụ thuộc vào yếu tố ngoài.

## M2.5 Các feature numeric tương quan ra sao?

In [ ]:
# TODO M2.5: correlation heatmap
plt.figure(figsize=(10,8))
corr= df_clean.select_dtypes(include=np.number).corr()
sns.heatmap(
    corr,
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)
plt.title('Correlation Matrix of Numeric Features')
plt.show()

**Nhận xét:** 
- engine_size, horsepower và curb_weight có tương quan dương khá mạnh với price.
- một số còn lại có hệ số tương quan thấp, mức độ liên hệ tuyến tính không đáng kể.

## M2.6 Biểu đồ tự chọn

Đặt một câu hỏi mới, chọn chart phù hợp và không lặp nguyên năm biểu đồ trên.
Xe sử dụng loại nhiên liệu khác nhau có mức giá khác nhau không?

In [ ]:
# TODO M2.6: biểu đồ tự chọn
plt.figure(figsize=(8,5))
avg_price = (
    df_clean
    .groupby('fuel_type')['price']
    .mean()
    .reset_index()
)
sns.barplot(data=avg_price, x='fuel_type', y='price')
plt.title('Average Car Price by Fuel Type')
plt.xlabel('Fuel Type')
plt.ylabel('Average Price (USD)')
plt.show()

**Nhận xét:** 
Nhiên liệu khác nhau có ảnh hưởng đến giá bán vì quan sát biểu đồ, nhóm diesel có xu hướng tập trung ở mức giá khác với nhóm gas.

# Tổng hợp

Viết:

- 3--5 phát hiện chính có dẫn chứng;
- ít nhất 2 hạn chế của dataset;
- một ví dụ về correlation không đồng nghĩa causation;
- một câu hỏi nên phân tích tiếp.

## Tổng hợp của sinh viên
Sau khi thực hiện 1 loạt biểu đồ, có thể rút ra nhận xét:
- Biểu đồ histogram cho thấy biến price có phân phối lệch phải, phần lớn xe trong dataset có giá ở mức thấp và trung bình, phần còn lại chỉ có 1 số ít xe có giá rất cao. 
- Biểu đồ countplot của body_style cho thấy số lượng xe giữa các nhóm không đồng đều, một số kiểu thân xe xuất hiện nhiều hơn các nhóm còn lại.
- Boxplot giữa price và body_style có sự khác biệt giá xe của các kiểu thân xe.
- Engine_size và price có biểu đồ scatter có xu hướng tăng cùng nhau, xe có động cơ lớn thường có giá cao hơn. 
- Heatmap tương quan cho thấy các biến như engine_size, horsepower và curb_weight có mức tương quan khá mạnh với price.
Bên cạnh đó, bộ dữ liệu vẫn có một số hạn chế. Một số nhóm body_style có số lượng mẫu khá ít nên kết quả phân tích chưa thật sự đúng và đáng tin cậy. Và bộ dữ liệu chỉ hoạt động trên 1 tệp nhỏ, không bao quát toàn bộ thị trường.
Correlation không đồng nghĩa với causation vì theo như khảo sát cho thấy: engine_size có tương quan dương với price, nhưng không có nghĩa rằng tăng kích thước động cơ sẽ thực tiếp làm giá xe tăng, có thể chịu ảnh hưởng của định giá bởi nhà sản xuất hay một vấn đề nào đó liên quan đến chất lượng xe.
Một câu hỏi nên phân tích tiếp: hãng xe có ảnh hưởng đến giá bán không? Và hãng nào có mức giá trung bình cao nhất trong dataset?